# TMDb Data Profile

This notebook documents the data-quality checks performed on the candidate movie list and the TMDb responses. It records the issues found, their effect on movie coverage, and the evidence behind decisions used in the pipeline.

## 1. Candidate ID quality

The supplied CSV contains one column of candidate TMDb movie IDs. The first check accounts for every input row and identifies which distinct IDs can be sent to TMDb.

In [5]:
from pathlib import Path

import pandas as pd

# Supports starting the notebook from the project root or Data Profile.
project_root = Path.cwd()
if project_root.name.lower() == "data profile":
    project_root = project_root.parent

audit_path = project_root / "data" / "validated" / "id_audit.csv"
id_audit = pd.read_csv(audit_path, dtype={"tmdb_id": str}, keep_default_na=False)

status_order = ["missing", "invalid_format", "duplicate", "ready_to_fetch"]
counts = id_audit["status"].value_counts().reindex(status_order, fill_value=0)

assert counts.sum() == len(id_audit), "The audit has an unexpected status."

print(f"Total input rows: {len(id_audit)}")
display(counts.rename("rows").to_frame())

Total input rows: 127


,rows
status,
missing,1
invalid_format,3
duplicate,6
ready_to_fetch,117


In [6]:
issues = id_audit.loc[
    id_audit["status"].ne("ready_to_fetch"),
    ["source_line", "tmdb_id", "status"],
].copy()

# Quotes make the blank value visible as '' in the notebook.
issues["tmdb_id"] = issues["tmdb_id"].map(repr)
display(issues)

,source_line,tmdb_id,status
28,30,'-1',invalid_format
41,43,'603',duplicate
58,60,'278',duplicate
67,69,'',missing
88,90,'1891',duplicate
89,91,'155',duplicate
92,94,'550',duplicate
93,95,'tt0468569',invalid_format
100,102,'680',duplicate
109,111,'N/A',invalid_format


### Finding and decision

Of the 127 input rows, 1 was blank, 3 failed the positive-integer rule,
and 6 were repeat occurrences of IDs already present in the list.
The malformed values were `-1`, `tt0468569`, and `N/A`.

The pipeline kept the first occurrence of each eligible ID. That left
117 distinct IDs to request from TMDb. The six duplicate rows added no
new IDs. The four unusable rows reduced coverage of the supplied list.

Reconciliation: 127 = 1 blank + 3 invalid + 6 duplicates + 117 ready to fetch.

## 2. TMDb response structure

I inspected the cached details and credits for The Dark Knight (TMDb ID 155).
The goal was to see how TMDb represents movie attributes, collection
membership, genres, cast, and crew before deciding how to flatten them.

In [7]:
import json
import sqlite3
from contextlib import closing

cache_path = project_root / "data" / "cache" / "tmdb.sqlite"
movie_id = "155"

if not cache_path.is_file():
    raise FileNotFoundError(f"Run pipeline/fetch.py first. Cache missing at {cache_path}")

with closing(sqlite3.connect(cache_path)) as connection:
    rows = connection.execute(
        """
        SELECT endpoint, payload
        FROM responses
        WHERE movie_id = ?
        """,
        (movie_id,),
    ).fetchall()

responses = {endpoint: json.loads(payload) for endpoint, payload in rows}
assert responses.keys() == {"details", "credits"}, f"Missing response for {movie_id}"

details = responses["details"]
credits = responses["credits"]
assert str(details["id"]) == str(credits["id"]) == movie_id

print(f"Movie: {details['title']} ({movie_id})")
print("Details fields:", list(details))
print("Credits fields:", list(credits))

Movie: The Dark Knight (155)
Details fields: ['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']
Credits fields: ['id', 'cast', 'crew']


In [8]:
collection = details["belongs_to_collection"]
first_billed_actor = min(credits["cast"], key=lambda actor: actor["order"])
directors = [person for person in credits["crew"] if person["job"] == "Director"]

print("Collection:", collection["id"], collection["name"])
print("Genres:", [genre["name"] for genre in details["genres"]])
print("First-billed actor:", first_billed_actor["name"])
print("First crew entry:", credits["crew"][0]["name"], credits["crew"][0]["job"])
print("Directors:", [person["name"] for person in directors])

Collection: 263 The Dark Knight Collection
Genres: ['Action', 'Thriller', 'Crime']
First-billed actor: Christian Bale
First crew entry: Jonathan Nolan Screenplay
Directors: ['Christopher Nolan']


### Finding and decision

The details response contains single movie values such as title and release
date, a nested collection object, and a list of genres. The credits response
contains separate cast and crew lists. Both responses share the same movie ID.

The main movie table will have one row per movie. Collection ID and name can
be extracted into that row. If we need genre-level analysis, each movie–genre
pair can go into a separate table so a movie with three genres is counted once
in movie-level totals.

For credits, billing order identifies the first-billed actor in this example.
The crew list includes several jobs, so directors are identified using the
Director job label. This example shows the structure. Coverage across the full
candidate list is checked in the next section.

## 3. Acquisition coverage

I compared the validated ID list with the fetch audit to account for every
requested movie. I checked movie details and credits separately because
a movie with details can still be used in analyses that do not need credits.

In [9]:
validated_path = project_root / "data" / "validated" / "validated_ids.csv"
fetch_audit_path = project_root / "data" / "cache" / "fetch_audit.csv"

validated = pd.read_csv(validated_path, dtype={"tmdb_id": str})
fetch_audit = pd.read_csv(fetch_audit_path, dtype={"tmdb_id": str})

assert validated["tmdb_id"].is_unique, "Duplicate IDs in the validated list."
assert fetch_audit["tmdb_id"].is_unique, "Duplicate IDs in the fetch audit."
assert set(validated["tmdb_id"]) == set(fetch_audit["tmdb_id"]), (
    "The fetch audit and validated list contain different IDs."
)

details_ok = fetch_audit["details"].eq("ok")
credits_ok = fetch_audit["credits"].eq("ok")

coverage = pd.Series({
    "Both responses available": (details_ok & credits_ok).sum(),
    "Details only": (details_ok & ~credits_ok).sum(),
    "Credits only": (~details_ok & credits_ok).sum(),
    "Neither response available": (~details_ok & ~credits_ok).sum(),
}, name="movies")

assert coverage.sum() == len(validated)
display(coverage.to_frame())

print(f"Movies with details available: {details_ok.sum()}/{len(validated)}")

,movies
Both responses available,112
Details only,0
Credits only,0
Neither response available,5


Movies with details available: 112/117


In [10]:
#incomplete movies
incomplete = fetch_audit.loc[
    ~(details_ok & credits_ok),
    ["tmdb_id", "details", "credits"],
]

display(incomplete)

,tmdb_id,details,credits
16,8471293,8471293 details: HTTP 404,8471293 credits: HTTP 404
32,9412887,9412887 details: HTTP 404,9412887 credits: HTTP 404
67,8471294,8471294 details: HTTP 404,8471294 credits: HTTP 404
85,9412888,9412888 details: HTTP 404,9412888 credits: HTTP 404
89,7733901,7733901 details: HTTP 404,7733901 credits: HTTP 404


### Finding and decision

Of the 117 validated movie IDs, 112 had both details and credits.
Five IDs had at least one unsuccessful request. The table above records
which endpoint failed for each ID.

Movies with available details can proceed to movie-level profiling. Movies
with missing credits are excluded from analyses that require director or
lead-actor information. IDs with missing details are excluded from analyses
that require movie attributes.

## 4. Movie details quality

I profiled the cached details for every candidate with a successful details
request. I checked for missing or zero financial values and unusable release
dates before deciding which movies were eligible for each analysis.

In [11]:
expected_ids = set(fetch_audit.loc[details_ok, "tmdb_id"])

with closing(sqlite3.connect(cache_path)) as connection:
    rows = connection.execute("""
        SELECT movie_id, payload
        FROM responses
        WHERE endpoint = 'details'
    """).fetchall()

records = [
    {"tmdb_id": movie_id, **json.loads(payload)}
    for movie_id, payload in rows
    if movie_id in expected_ids
]

movies = pd.DataFrame(records)

assert movies["tmdb_id"].is_unique
assert set(movies["tmdb_id"]) == expected_ids, "Cache and fetch audit disagree."

print(f"Movies profiled: {len(movies)}")
display(movies[["tmdb_id", "title", "release_date", "budget", "revenue"]].head())

Movies profiled: 112


,tmdb_id,title,release_date,budget,revenue
0,155,The Dark Knight,2008-07-16,185000000,1008480529
1,165,Back to the Future Part II,1989-11-22,40000000,332861197
2,242,The Godfather Part III,1990-12-25,54000000,136766062
3,49026,The Dark Knight Rises,2012-07-17,250000000,1081041287
4,8077,Alien³,1992-05-22,50000000,159773545


In [12]:
budget = pd.to_numeric(movies["budget"], errors="coerce")
revenue = pd.to_numeric(movies["revenue"], errors="coerce")
release_date = pd.to_datetime(
    movies["release_date"], format="%Y-%m-%d", errors="coerce"
)

checks = pd.DataFrame({
    "Budget missing or non-numeric": budget.isna(),
    "Budget is zero": budget.eq(0),
    "Revenue missing or non-numeric": revenue.isna(),
    "Revenue is zero": revenue.eq(0),
    "Release date missing or invalid": release_date.isna(),
})

display(checks.sum().rename("movies_affected").to_frame())

affected = checks.any(axis=1)
print(f"Distinct movies affected: {affected.sum()}")
display(
    movies.loc[
        affected, ["tmdb_id", "title", "release_date", "budget", "revenue"]
    ].join(checks.loc[affected])
)

,movies_affected
Budget missing or non-numeric,0
Budget is zero,0
Revenue missing or non-numeric,0
Revenue is zero,0
Release date missing or invalid,0


Distinct movies affected: 0


,tmdb_id,title,release_date,budget,revenue,Budget missing or non-numeric,Budget is zero,Revenue missing or non-numeric,Revenue is zero,Release date missing or invalid


### Finding

All movies with cached details had parseable release dates and numeric,
nonzero budgets and revenues. These five checks flagged zero movies.
I then checked for negative amounts and reviewed the highest and lowest
revenue-to-budget ratios.

In [13]:
print(f"Negative budgets: {budget.lt(0).sum()}")
print(f"Negative revenues: {revenue.lt(0).sum()}")

positive_amounts = budget.gt(0) & revenue.gt(0)
ratios = movies.loc[positive_amounts, ["tmdb_id", "title"]].copy()
ratios["budget"] = budget.loc[positive_amounts]
ratios["revenue"] = revenue.loc[positive_amounts]
ratios["revenue_to_budget"] = ratios["revenue"] / ratios["budget"]

print("\nFive highest ratios")
display(ratios.nlargest(5, "revenue_to_budget"))

print("\nFive lowest ratios")
display(ratios.nsmallest(5, "revenue_to_budget"))

Negative budgets: 0
Negative revenues: 0

Five highest ratios


,tmdb_id,title,budget,revenue,revenue_to_budget
23,11,Star Wars,11000000,775398007,70.490728
85,578,Jaws,7000000,470653000,67.236143
59,238,The Godfather,6000000,245066411,40.844402
67,429,"The Good, the Bad and the Ugly",1200000,38900000,32.416667
49,1891,The Empire Strikes Back,18000000,538400000,29.911111



Five lowest ratios


,tmdb_id,title,budget,revenue,revenue_to_budget
24,278,The Shawshank Redemption,25000000,28341469,1.133659
56,78,Blade Runner,28000000,41722424,1.490087
71,550,Fight Club,63000000,100853753,1.600853
55,641,Requiem for a Dream,4500000,7390108,1.642246
102,534,Terminator Salvation,200000000,371353001,1.856765


In [14]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pipeline.transform import flatten_movie_details

flat_movies = flatten_movie_details(movies)

assert flat_movies["tmdb_id"].is_unique
assert flat_movies["tmdb_id"].equals(movies["tmdb_id"])

print(f"Movies before flattening: {len(movies)}")
print(f"Movies after flattening:  {len(flat_movies)}")

display(
    flat_movies.loc[
        flat_movies["tmdb_id"].isin(["155", "278"]),
        ["tmdb_id", "title", "collection_id", "collection_name", "revenue_to_budget"],
    ]
)

Movies before flattening: 112
Movies after flattening:  112


,tmdb_id,title,collection_id,collection_name,revenue_to_budget
0,155,The Dark Knight,263,The Dark Knight Collection,5.451246
24,278,The Shawshank Redemption,<NA>,NaN,1.133659


In [16]:
#franchise vs standalone movies
profile = flat_movies.assign(
    movie_group=flat_movies["collection_id"].notna().map({
        True: "Franchise",
        False: "Standalone",
    })
)

windows = {
    "All years": profile,
    "2000 onward": profile.loc[profile["release_date"].dt.year.ge(2000)],
}

coverage = pd.concat({
    window: films.groupby("movie_group").agg(
        films=("tmdb_id", "size"),
        films_with_ratio=("revenue_to_budget", "count"),
        median_ratio=("revenue_to_budget", "median"),
    )
    for window, films in windows.items()
}).rename_axis(["time_window", "movie_group"])

display(coverage)

films  films_with_ratio  median_ratio
time_window movie_group                                       
All years   Franchise       96                96      5.623799
            Standalone      16                16      3.844812
2000 onward Franchise       63                63      5.051939
            Standalone       8                 8      3.844812

Franchise films had a higher median revenue-to-budget ratio in both time windows. The gap narrowed from 1.78 to 1.21 after restricting the comparison to films released from 2000 onward. The later comparison includes 63 franchise films and 8 standalones, so its standalone estimate is based on a small sample.

In [17]:
comparison = []

for window, films in windows.items():
    franchise_films = films.loc[films["collection_id"].notna()]
    standalone_films = films.loc[films["collection_id"].isna()]

    collections = (
        franchise_films.groupby("collection_id")
        .agg(
            collection_name=("collection_name", "first"),
            films_in_sample=("tmdb_id", "size"),
            median_ratio=("revenue_to_budget", "median"),
        )
        .sort_values("films_in_sample", ascending=False)
    )

    print(window)
    display(collections)

    comparison.append({
        "time_window": window,
        "collections_with_ratio": collections["median_ratio"].count(),
        "franchise_median_equal_weight": collections["median_ratio"].median(),
        "standalone_median": standalone_films["revenue_to_budget"].median(),
    })

display(pd.DataFrame(comparison).set_index("time_window").round(2))

All years


,collection_name,films_in_sample,median_ratio
collection_id,,,
1241,Harry Potter Collection,8,6.165086
10,Star Wars Collection,6,13.401690
295,Pirates of the Caribbean Collection,5,3.460532
87359,Mission: Impossible Collection,5,4.551444
2150,Shrek Collection,4,5.634445
1570,Die Hard Collection,4,3.777223
528,The Terminator Collection,4,3.625805
84,Indiana Jones Collection,4,10.887634
10194,Toy Story Collection,4,5.831315


2000 onward


,collection_name,films_in_sample,median_ratio
collection_id,,,
1241,Harry Potter Collection,8,6.165086
295,Pirates of the Caribbean Collection,5,3.460532
2150,Shrek Collection,4,5.634445
87359,Mission: Impossible Collection,4,4.461275
86311,The Avengers Collection,4,6.872545
263,The Dark Knight Collection,3,4.324165
304,Ocean's Collection,3,3.662501
119,The Lord of the Rings Collection,3,11.725157
121938,The Hobbit Collection,3,3.833600


,collections_with_ratio,franchise_median_equal_weight,standalone_median
time_window,,,
All years,32,5.73,3.84
2000 onward,24,4.39,3.84


In [18]:
import importlib
import pipeline.transform

importlib.reload(pipeline.transform)
from pipeline.transform import flatten_lead_actors

with closing(sqlite3.connect(cache_path)) as connection:
    credit_rows = connection.execute("""
        SELECT movie_id, payload
        FROM responses
        WHERE endpoint = 'credits'
    """).fetchall()

movie_ids = set(flat_movies["tmdb_id"])
credits_by_movie = {
    movie_id: json.loads(payload)
    for movie_id, payload in credit_rows
    if movie_id in movie_ids
}

lead_actors = flatten_lead_actors(credits_by_movie)

movies_with_leads = flat_movies.merge(
    lead_actors, on="tmdb_id", how="left", validate="one_to_one"
)

assert len(movies_with_leads) == len(flat_movies)

print(f"Movies with a first-billed actor: {len(lead_actors)}/{len(flat_movies)}")
display(
    movies_with_leads.loc[
        movies_with_leads["tmdb_id"].eq("155"),
        ["tmdb_id", "title", "lead_actor_name"],
    ]
)

ImportError: cannot import name 'flatten_lead_actors' from 'pipeline.transform' (c:\Users\User\OneDrive\Desktop\amal\lab2\pipeline\transform.py)